# Predicción de Churn en Telecomunicaciones
## Metodología CRISP-DM aplicada al dataset IBM Telco Customer Churn (Extended)

---
**Objetivo de negocio:** Identificar clientes con alta probabilidad de cancelar su suscripción antes de que ocurra el abandono, para activar campañas de retención personalizadas y reducir la tasa de churn en al menos 15%.

**Estructura del notebook:**
1. Comprensión del negocio
2. Comprensión de los datos
3. Preparación de datos y Feature Engineering (énfasis principal)
4. Modelado
5. Evaluación
6. Conclusiones y despliegue


---
## Fase 1 — Comprensión del Negocio

### Contexto
Una empresa de telecomunicaciones ficticia enfrenta una tasa de churn del ~26%. Retener un cliente existente cuesta entre 5 y 7 veces menos que adquirir uno nuevo. El equipo de retención actúa de forma reactiva, sin anticiparse a la salida del cliente.

### Objetivos estratégicos
| Objetivo | Métrica | Meta |
|---|---|---|
| Reducir churn | Tasa de abandono mensual | ↓ 15% vs año anterior |
| Identificar churners | Recall clase positiva | ≥ 75% |
| Rentabilidad | ROI campañas retención | ≥ 3:1 |
| Calidad del modelo | ROC-AUC en test | ≥ 0.85 |

### Definición de éxito del modelo
- **Métrica principal:** ROC-AUC (maximiza discriminación general)
- **Métrica operativa:** Recall ≥ 0.75 en clase Churn=1 (preferimos falsos positivos sobre falsos negativos)
- **Umbral de decisión:** ajustable según costo de campaña de retención


In [ ]:
# ============================================================
# INSTALACIÓN DE DEPENDENCIAS
# ============================================================
# Ejecutar si no están instaladas
# !pip install pandas numpy matplotlib seaborn scikit-learn xgboost imbalanced-learn shap plotly


In [ ]:
# ============================================================
# IMPORTACIONES GENERALES
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

print('✅ Librerías cargadas correctamente')


---
## Fase 2 — Comprensión de los Datos


In [ ]:
# ============================================================
# 2.1 CARGA DE DATOS — IBM Telco Extended
# ============================================================
# El dataset Extended incluye variables adicionales:
# Satisfaction Score, Churn Reason, Churn Category, CLTV,
# Number of Referrals, Avg Monthly Long Distance Charges, etc.

# Opción A: Kaggle (requiere cuenta)
# kaggle datasets download -d ylchang/telco-customer-churn-1113

# Opción B: Carga directa desde URL pública
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

# Cargamos la versión estándar como base (Extended usa las mismas columnas core)
df_raw = pd.read_csv(URL)

# Simulamos columnas del Extended que no están en la versión base
# (en producción, estas vendrían del CSV de Kaggle)
np.random.seed(42)
n = len(df_raw)

df_raw['SatisfactionScore']        = np.random.randint(1, 6, n)          # 1-5
df_raw['NumberOfReferrals']        = np.random.randint(0, 11, n)         # 0-10
df_raw['AvgMonthlyLongDistCharge'] = np.random.uniform(0, 50, n).round(2)
df_raw['CLTV']                     = np.random.randint(2000, 8000, n)    # Customer Lifetime Value
df_raw['Latitude']                 = np.random.uniform(32, 42, n).round(4)
df_raw['Longitude']                = np.random.uniform(-120, -70, n).round(4)

# Hacemos que SatisfactionScore bajo se correlacione más con churn
churn_mask = df_raw['Churn'] == 'Yes'
df_raw.loc[churn_mask, 'SatisfactionScore'] = np.random.choice([1, 2, 3], churn_mask.sum())

df = df_raw.copy()
print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
df.head(3)


In [ ]:
# ============================================================
# 2.2 ESTRUCTURA Y TIPOS DE DATOS
# ============================================================
print('=== INFORMACIÓN GENERAL ===')
print(f'Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}')
print()

# Clasificar columnas por tipo
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f'Numéricas ({len(num_cols)}): {num_cols}')
print(f'Categóricas ({len(cat_cols)}): {cat_cols}')
print()
df.dtypes


In [ ]:
# ============================================================
# 2.3 ANÁLISIS DE VALORES NULOS Y PROBLEMA CONOCIDO
# ============================================================

# Problema conocido: TotalCharges tiene espacios en blanco
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

nulls = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(2)
null_df = pd.DataFrame({'Nulos': nulls, 'Porcentaje %': nulls_pct})
null_df = null_df[null_df['Nulos'] > 0]

print('=== VALORES NULOS ===')
print(null_df)
print()
print('Los 11 nulos en TotalCharges corresponden a clientes con tenure=0')
print(df[df['TotalCharges'].isnull()][['tenure', 'MonthlyCharges', 'TotalCharges']].head())


In [ ]:
# ============================================================
# 2.4 DISTRIBUCIÓN DE LA VARIABLE OBJETIVO (CHURN)
# ============================================================
churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Barplot
axes[0].bar(churn_counts.index, churn_counts.values,
            color=['#5DCAA5', '#E8593C'], width=0.5)
axes[0].set_title('Distribución de Churn (conteo)', fontsize=13)
axes[0].set_ylabel('Número de clientes')
for i, (v, p) in enumerate(zip(churn_counts.values, churn_pct.values)):
    axes[0].text(i, v + 50, f'{v:,}\n({p:.1f}%)', ha='center', fontsize=11)

# Pie
axes[1].pie(churn_counts.values, labels=churn_counts.index,
            colors=['#5DCAA5', '#E8593C'], autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Proporción Churn vs No Churn', fontsize=13)

plt.suptitle('Desbalance de clases — tasa de churn ~26%', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f'Ratio desbalance: {churn_pct["No"]:.1f}% No Churn | {churn_pct["Yes"]:.1f}% Churn')


In [ ]:
# ============================================================
# 2.5 EDA — VARIABLES NUMÉRICAS
# ============================================================
num_features = ['tenure', 'MonthlyCharges', 'TotalCharges',
                'SatisfactionScore', 'NumberOfReferrals',
                'AvgMonthlyLongDistCharge', 'CLTV']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(num_features):
    for label, color in [('No', '#5DCAA5'), ('Yes', '#E8593C')]:
        subset = df[df['Churn'] == label][col].dropna()
        axes[i].hist(subset, bins=30, alpha=0.6, color=color,
                     label=f'Churn={label}', density=True)
    axes[i].set_title(col, fontsize=11)
    axes[i].legend(fontsize=8)
    axes[i].set_xlabel('')

axes[-1].set_visible(False)
plt.suptitle('Distribución de variables numéricas por Churn', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 2.6 EDA — VARIABLES CATEGÓRICAS CLAVE
# ============================================================
cat_features = ['Contract', 'PaymentMethod', 'InternetService',
                'tenure_group']

# Creamos tenure_group temporal para visualización
df['tenure_group'] = pd.cut(df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['0-1 año', '1-2 años', '2-4 años', '4+ años'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ['Contract', 'PaymentMethod', 'InternetService']):
    churn_rate = df.groupby(col)['Churn'].apply(
        lambda x: (x == 'Yes').sum() / len(x) * 100
    ).sort_values(ascending=False)
    
    bars = ax.barh(churn_rate.index, churn_rate.values,
                   color='#E8593C', alpha=0.75)
    ax.set_xlabel('Tasa de churn (%)')
    ax.set_title(f'Churn % por {col}', fontsize=12)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    
    for bar, val in zip(bars, churn_rate.values):
        ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=9)

plt.suptitle('Tasa de churn por variable categórica', fontsize=14)
plt.tight_layout()
plt.show()

# Eliminar columna temporal
df.drop(columns=['tenure_group'], inplace=True)


In [ ]:
# ============================================================
# 2.7 MATRIZ DE CORRELACIÓN (variables numéricas)
# ============================================================
# Codificación temporal de Churn para correlación
df_corr = df[num_features + ['Churn']].copy()
df_corr['Churn_bin'] = (df_corr['Churn'] == 'Yes').astype(int)
df_corr = df_corr.drop(columns='Churn').dropna()

corr_matrix = df_corr.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
plt.figure(figsize=(10, 7))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            linewidths=0.5, annot_kws={'size': 9})
plt.title('Matriz de correlación — variables numéricas vs Churn', fontsize=13)
plt.tight_layout()
plt.show()

# Correlaciones con Churn ordenadas
print('=== Correlación con Churn (Churn_bin) ===')
print(corr_matrix['Churn_bin'].drop('Churn_bin').sort_values(key=abs, ascending=False))


---
## Fase 3 — Preparación de Datos y Feature Engineering

Esta es la fase más extensa. Se divide en:
1. Limpieza base
2. Tratamiento de la variable objetivo
3. Feature engineering de comportamiento
4. Feature engineering de riesgo compuesto
5. Feature engineering de interacciones
6. Encoding de categóricas
7. Tratamiento del desbalance
8. Split train/test y escalado


In [ ]:
# ============================================================
# 3.1 LIMPIEZA BASE
# ============================================================
df_clean = df.copy()

# Convertir TotalCharges a numérico
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')

# Imputar nulos en TotalCharges con 0 (clientes con tenure=0)
df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(0)

# Eliminar columna customerID (identificador, no predictivo)
df_clean.drop(columns=['customerID'], inplace=True)

print(f'Nulos restantes: {df_clean.isnull().sum().sum()}')
print(f'Shape después de limpieza: {df_clean.shape}')


In [ ]:
# ============================================================
# 3.2 VARIABLE OBJETIVO
# ============================================================
df_clean['Churn'] = (df_clean['Churn'] == 'Yes').astype(int)

print(f'Distribución Churn codificado:')
print(df_clean['Churn'].value_counts())


In [ ]:
# ============================================================
# 3.3 FEATURE ENGINEERING — COMPORTAMIENTO TEMPORAL
# ============================================================

# 1. Segmento de antigüedad (tenure cohort)
df_clean['tenure_cohort'] = pd.cut(
    df_clean['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=[0, 1, 2, 3],  # 0=nuevo, 3=leal
    include_lowest=True
).astype(int)

# 2. Costo por mes promedio real (TotalCharges / tenure)
df_clean['avg_monthly_spend'] = np.where(
    df_clean['tenure'] > 0,
    df_clean['TotalCharges'] / df_clean['tenure'],
    df_clean['MonthlyCharges']
).round(2)

# 3. Diferencia entre cargo actual y gasto histórico promedio
# Un aumento brusco puede indicar insatisfacción
df_clean['charge_drift'] = (df_clean['MonthlyCharges'] - df_clean['avg_monthly_spend']).round(2)

# 4. Meses restantes para vencimiento de contrato estimado
contract_duration_map = {'Month-to-month': 1, 'One year': 12, 'Two year': 24}
df_clean['contract_duration_months'] = df_clean['Contract'].map(contract_duration_map)
df_clean['months_to_contract_end'] = np.maximum(
    df_clean['contract_duration_months'] - (df_clean['tenure'] % df_clean['contract_duration_months']),
    0
)

# 5. Es cliente nuevo (primeros 6 meses — mayor riesgo de churn temprano)
df_clean['is_new_customer'] = (df_clean['tenure'] <= 6).astype(int)

print('✅ Features temporales creadas: tenure_cohort, avg_monthly_spend, charge_drift,'
      ' contract_duration_months, months_to_contract_end, is_new_customer')


In [ ]:
# ============================================================
# 3.4 FEATURE ENGINEERING — ADOPCIÓN DE SERVICIOS
# ============================================================

# Mapeamos Yes/No/No internet service a valores numéricos
yes_no_map = {'Yes': 1, 'No': 0, 'No internet service': 0, 'No phone service': 0}

service_cols = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'MultipleLines'
]

for col in service_cols:
    df_clean[col + '_bin'] = df_clean[col].map(yes_no_map).fillna(0).astype(int)

# 1. Número total de servicios adicionales contratados
df_clean['num_additional_services'] = df_clean[[c + '_bin' for c in service_cols]].sum(axis=1)

# 2. Ratio de servicios vs máximo posible (7 servicios)
df_clean['service_adoption_rate'] = (df_clean['num_additional_services'] / 7).round(3)

# 3. Tiene servicios de seguridad (menor churn esperado)
df_clean['has_security_services'] = (
    (df_clean['OnlineSecurity_bin'] == 1) |
    (df_clean['OnlineBackup_bin'] == 1) |
    (df_clean['DeviceProtection_bin'] == 1)
).astype(int)

# 4. Es solo cliente de streaming (alto riesgo — puede cambiarse a Netflix/otros)
df_clean['streaming_only'] = (
    (df_clean['StreamingTV_bin'] == 1) |
    (df_clean['StreamingMovies_bin'] == 1)
).astype(int)

print('✅ Features de servicios: num_additional_services, service_adoption_rate,'
      ' has_security_services, streaming_only')


In [ ]:
# ============================================================
# 3.5 FEATURE ENGINEERING — SCORE DE RIESGO COMPUESTO
# ============================================================
# Combinamos señales individuales en un score de riesgo interpretable

# Normalización min-max simple para componentes del score
def minmax_norm(series):
    return (series - series.min()) / (series.max() - series.min() + 1e-9)

# Componentes del riesgo (0-1 cada uno, mayor = más riesgo)
risk_low_satisfaction   = minmax_norm(5 - df_clean['SatisfactionScore'])  # invertido
risk_high_monthly_charge = minmax_norm(df_clean['MonthlyCharges'])
risk_short_tenure        = minmax_norm(1 / (df_clean['tenure'] + 1))       # invertido
risk_month_to_month      = (df_clean['Contract'] == 'Month-to-month').astype(float)
risk_low_services        = minmax_norm(1 - df_clean['service_adoption_rate'])
risk_new_customer        = df_clean['is_new_customer'].astype(float)

# Score ponderado (pesos basados en importancia esperada del negocio)
df_clean['churn_risk_score'] = (
    0.25 * risk_low_satisfaction    +  # satisfacción es el mayor predictor
    0.20 * risk_month_to_month      +  # contrato mensual = fácil salida
    0.20 * risk_short_tenure        +  # clientes nuevos son más volátiles
    0.15 * risk_high_monthly_charge +  # cargos altos generan fricción
    0.10 * risk_low_services        +  # poca adopción = poca adhesión
    0.10 * risk_new_customer           # cliente nuevo (<6 meses)
).round(4)

# Segmento de riesgo para comunicación al negocio
df_clean['risk_segment'] = pd.cut(
    df_clean['churn_risk_score'],
    bins=[0, 0.33, 0.66, 1.0],
    labels=['Bajo', 'Medio', 'Alto'],
    include_lowest=True
)

# Visualización del score vs churn real
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución del score por clase
for label, color in [(0, '#5DCAA5'), (1, '#E8593C')]:
    subset = df_clean[df_clean['Churn'] == label]['churn_risk_score']
    axes[0].hist(subset, bins=40, alpha=0.65, color=color,
                 label=f'Churn={label}', density=True)
axes[0].set_title('Score de riesgo compuesto vs Churn real', fontsize=12)
axes[0].set_xlabel('Churn Risk Score')
axes[0].legend()

# Tasa de churn por segmento
seg_churn = df_clean.groupby('risk_segment', observed=True)['Churn'].mean() * 100
colors_seg = ['#5DCAA5', '#EF9F27', '#E8593C']
axes[1].bar(seg_churn.index, seg_churn.values, color=colors_seg, width=0.5)
axes[1].set_title('Tasa de churn real por segmento de riesgo', fontsize=12)
axes[1].set_ylabel('Tasa de churn (%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
for i, v in enumerate(seg_churn.values):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=11)

plt.suptitle('Validación del Score de Riesgo Compuesto', fontsize=13)
plt.tight_layout()
plt.show()

print('✅ churn_risk_score y risk_segment creados')


In [ ]:
# ============================================================
# 3.6 FEATURE ENGINEERING — INTERACCIONES Y RATIOS
# ============================================================

# 1. Gasto total por servicio adicional (valor percibido)
df_clean['spend_per_service'] = np.where(
    df_clean['num_additional_services'] > 0,
    df_clean['MonthlyCharges'] / (df_clean['num_additional_services'] + 1),
    df_clean['MonthlyCharges']
).round(2)

# 2. CLTV por mes (rentabilidad mensual estimada)
df_clean['cltv_per_month'] = np.where(
    df_clean['tenure'] > 0,
    df_clean['CLTV'] / df_clean['tenure'],
    df_clean['CLTV']
).round(2)

# 3. Ratio referidos / antigüedad (clientes muy satisfechos refieren más)
df_clean['referral_rate'] = np.where(
    df_clean['tenure'] > 0,
    df_clean['NumberOfReferrals'] / df_clean['tenure'],
    0
).round(4)

# 4. Interacción: cliente senior con contrato mensual (grupo de alto riesgo)
df_clean['SeniorCitizen'] = df_clean['SeniorCitizen'].astype(int)
df_clean['senior_month_to_month'] = (
    (df_clean['SeniorCitizen'] == 1) &
    (df_clean['Contract'] == 'Month-to-month')
).astype(int)

# 5. Interacción: sin soporte técnico + cargo alto (frustración potencial)
df_clean['no_support_high_charge'] = (
    (df_clean['TechSupport_bin'] == 0) &
    (df_clean['MonthlyCharges'] > df_clean['MonthlyCharges'].median())
).astype(int)

# 6. Log de TotalCharges (reduce asimetría)
df_clean['log_total_charges'] = np.log1p(df_clean['TotalCharges']).round(4)

# 7. Cuadrado de tenure (captura efecto no lineal de la fidelidad)
df_clean['tenure_squared'] = (df_clean['tenure'] ** 2)

# 8. Satisfacción × Número de referidos (engagement compuesto)
df_clean['satisfaction_referral_idx'] = (
    df_clean['SatisfactionScore'] * df_clean['NumberOfReferrals']
)

print('✅ Features de interacción creadas:')
new_features = ['spend_per_service', 'cltv_per_month', 'referral_rate',
                'senior_month_to_month', 'no_support_high_charge',
                'log_total_charges', 'tenure_squared', 'satisfaction_referral_idx']
for f in new_features:
    print(f'  - {f}')


In [ ]:
# ============================================================
# 3.7 ENCODING DE VARIABLES CATEGÓRICAS
# ============================================================
df_model = df_clean.copy()

# Binarias directas (Label Encoding)
binary_map = {'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0}
for col in ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']:
    df_model[col] = df_model[col].map(binary_map)

# One-Hot Encoding para categóricas con múltiples valores
ohe_cols = ['Contract', 'PaymentMethod', 'InternetService']
df_model = pd.get_dummies(df_model, columns=ohe_cols, drop_first=False, dtype=int)

# Eliminar columnas originales ya procesadas o redundantes
cols_to_drop = service_cols + ['risk_segment', 'contract_duration_months']
df_model.drop(columns=cols_to_drop, inplace=True, errors='ignore')

print(f'Shape tras encoding: {df_model.shape}')
print(f'Total features: {df_model.shape[1] - 1} (excluye Churn)')
df_model.head(2)


In [ ]:
# ============================================================
# 3.8 RESUMEN DE FEATURES CREADAS
# ============================================================
feature_summary = pd.DataFrame({
    'Feature': [
        'tenure_cohort', 'avg_monthly_spend', 'charge_drift',
        'months_to_contract_end', 'is_new_customer',
        'num_additional_services', 'service_adoption_rate',
        'has_security_services', 'streaming_only',
        'churn_risk_score',
        'spend_per_service', 'cltv_per_month', 'referral_rate',
        'senior_month_to_month', 'no_support_high_charge',
        'log_total_charges', 'tenure_squared', 'satisfaction_referral_idx'
    ],
    'Categoría': [
        'Temporal', 'Temporal', 'Temporal', 'Temporal', 'Temporal',
        'Servicios', 'Servicios', 'Servicios', 'Servicios',
        'Riesgo compuesto',
        'Interacción', 'Interacción', 'Interacción',
        'Interacción', 'Interacción', 'Transformación',
        'Transformación', 'Interacción'
    ],
    'Hipótesis de negocio': [
        'Clientes nuevos tienen mayor riesgo de churn temprano',
        'Gasto real mensual difiere del cargo facturado',
        'Incrementos bruscos en cargo generan insatisfacción',
        'Cercanía al fin de contrato puede acelerar decisión de salida',
        'Primeros 6 meses son críticos para retención',
        'Más servicios = mayor adhesión a la plataforma',
        'Adopción baja indica poca dependencia del proveedor',
        'Servicios de seguridad crean dependencia tecnológica',
        'Clientes solo de streaming pueden irse a competidores',
        'Agregación de señales da mejor predicción que señales aisladas',
        'Relación precio/valor percibido afecta lealtad',
        'Clientes con alto CLTV por mes son más estratégicos',
        'Clientes que refieren están más comprometidos',
        'Adultos mayores con contrato flexible son grupo vulnerable',
        'Frustración técnica no resuelta acelera abandono',
        'Reduce asimetría en distribución de cargos totales',
        'Efecto de retención no lineal: mejora con el tiempo',
        'Satisfacción + referidos como indicador de NPS implícito'
    ]
})

print('=== RESUMEN DE FEATURE ENGINEERING ===')
print(f'Total features creadas: {len(feature_summary)}')
print()
print(feature_summary.to_string(index=False))


In [ ]:
# ============================================================
# 3.9 SPLIT TRAIN / TEST
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]:,} filas | Test: {X_test.shape[0]:,} filas')
print(f'Churn en train: {y_train.mean()*100:.1f}% | Churn en test: {y_test.mean()*100:.1f}%')


In [ ]:
# ============================================================
# 3.10 TRATAMIENTO DEL DESBALANCE — SMOTE
# ============================================================
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'Antes de SMOTE — Churn: {y_train.sum():,} | No Churn: {(y_train==0).sum():,}')
print(f'Después de SMOTE — Churn: {y_train_res.sum():,} | No Churn: {(y_train_res==0).sum():,}')
print(f'Nuevo balance: {y_train_res.mean()*100:.1f}% churn')


In [ ]:
# ============================================================
# 3.11 ESCALADO — StandardScaler
# ============================================================
# Aplicamos solo a features numéricas continuas
scaler = StandardScaler()

# Solo escalar para modelos sensibles a la escala (Logistic Regression)
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled  = scaler.transform(X_test)

print('✅ Escalado completado (StandardScaler fit sobre train, aplicado a test)')


---
## Fase 4 — Modelado

Comparamos 3 modelos:
1. **Regresión Logística** — línea base interpretable
2. **Random Forest** — ensemble robusto
3. **XGBoost** — gradient boosting de alto rendimiento


In [ ]:
# ============================================================
# 4.1 ENTRENAMIENTO DE LOS 3 MODELOS
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (roc_auc_score, f1_score, recall_score,
                              precision_score, accuracy_score,
                              classification_report, confusion_matrix,
                              roc_curve, precision_recall_curve)

# Modelo 1: Regresión Logística (requiere datos escalados)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train_res)

# Modelo 2: Random Forest (no requiere escalado)
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    min_samples_leaf=5, random_state=42, n_jobs=-1
)
rf.fit(X_train_res, y_train_res)

# Modelo 3: XGBoost
xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='logloss',
    random_state=42, n_jobs=-1
)
xgb.fit(X_train_res, y_train_res,
        eval_set=[(X_test, y_test)], verbose=False)

print('✅ Modelos entrenados')


In [ ]:
# ============================================================
# 4.2 EVALUACIÓN COMPARATIVA
# ============================================================
def evaluate_model(model, X_test, y_test, model_name, scaled=False):
    X = X_test_scaled if scaled else X_test
    y_pred  = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    
    return {
        'Modelo': model_name,
        'ROC-AUC': round(roc_auc_score(y_test, y_proba), 4),
        'F1':      round(f1_score(y_test, y_pred), 4),
        'Recall':  round(recall_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        '_proba': y_proba,
        '_pred':  y_pred
    }

results = [
    evaluate_model(lr,  X_test, y_test, 'Regresión Logística', scaled=True),
    evaluate_model(rf,  X_test, y_test, 'Random Forest'),
    evaluate_model(xgb, X_test, y_test, 'XGBoost'),
]

results_df = pd.DataFrame(results).drop(columns=['_proba', '_pred'])
print('=== COMPARACIÓN DE MODELOS ===')
print(results_df.to_string(index=False))
print()

# Identificar mejor modelo por ROC-AUC
best_idx = results_df['ROC-AUC'].idxmax()
print(f'🏆 Mejor modelo: {results_df.loc[best_idx, "Modelo"]} '
      f'(ROC-AUC = {results_df.loc[best_idx, "ROC-AUC"]})')


In [ ]:
# ============================================================
# 4.3 CURVAS ROC
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

colors = ['#7F77DD', '#1D9E75', '#E8593C']
models_info = [
    (lr,  'Regresión Logística', True),
    (rf,  'Random Forest', False),
    (xgb, 'XGBoost', False)
]

# Curva ROC
for (model, name, scaled), color in zip(models_info, colors):
    X = X_test_scaled if scaled else X_test
    y_proba = model.predict_proba(X)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Aleatorio')
axes[0].fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
axes[0].set_xlabel('Tasa de Falsos Positivos')
axes[0].set_ylabel('Tasa de Verdaderos Positivos (Recall)')
axes[0].set_title('Curva ROC — Comparación de modelos', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Curva Precision-Recall (más informativa con desbalance)
for (model, name, scaled), color in zip(models_info, colors):
    X = X_test_scaled if scaled else X_test
    y_proba = model.predict_proba(X)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    axes[1].plot(rec, prec, color=color, lw=2, label=name)

baseline = y_test.mean()
axes[1].axhline(baseline, color='gray', linestyle='--', alpha=0.6,
                label=f'Baseline ({baseline:.2f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 4.4 MATRICES DE CONFUSIÓN
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, (model, name, scaled) in zip(axes, models_info):
    X = X_test_scaled if scaled else X_test
    y_pred = model.predict(X)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                cmap='Blues', linewidths=0.5,
                xticklabels=['No Churn', 'Churn'],
                yticklabels=['No Churn', 'Churn'])
    ax.set_title(name, fontsize=12)
    ax.set_ylabel('Real')
    ax.set_xlabel('Predicho')

plt.suptitle('Matrices de Confusión en datos de prueba', fontsize=13)
plt.tight_layout()
plt.show()

# Reporte detallado del mejor modelo (XGBoost)
print('=== REPORTE DETALLADO — XGBoost ===')
print(classification_report(y_test, xgb.predict(X_test),
                             target_names=['No Churn', 'Churn']))


In [ ]:
# ============================================================
# 4.5 IMPORTANCIA DE FEATURES — Random Forest y XGBoost
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

top_n = 20

for ax, model, name in [
    (axes[0], rf,  'Random Forest'),
    (axes[1], xgb, 'XGBoost')
]:
    importances = pd.Series(
        model.feature_importances_, index=X_train_res.columns
    ).nlargest(top_n).sort_values()
    
    colors_imp = ['#E8593C' if 'churn_risk' in i or 'satisfaction' in i.lower()
                  else '#7F77DD' if any(k in i for k in ['tenure', 'charge', 'contract'])
                  else '#1D9E75'
                  for i in importances.index]
    
    ax.barh(importances.index, importances.values, color=colors_imp, height=0.65)
    ax.set_title(f'Top {top_n} features — {name}', fontsize=12)
    ax.set_xlabel('Importancia')
    
    # Leyenda de colores
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(color='#E8593C', label='Satisfacción / Riesgo'),
        Patch(color='#7F77DD', label='Temporales / Contrato'),
        Patch(color='#1D9E75', label='Servicios / Otros')
    ]
    ax.legend(handles=legend_elements, fontsize=9, loc='lower right')

plt.suptitle('Importancia de Features por Modelo', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 4.6 ANÁLISIS SHAP — INTERPRETABILIDAD (XGBoost)
# ============================================================
try:
    import shap
    
    explainer   = shap.TreeExplainer(xgb)
    shap_values = explainer.shap_values(X_test)
    
    plt.figure(figsize=(12, 8))
    shap.summary_plot(
        shap_values, X_test,
        plot_type='bar',
        max_display=20,
        show=False
    )
    plt.title('SHAP — Importancia global de features (XGBoost)', fontsize=13)
    plt.tight_layout()
    plt.show()
    
    plt.figure(figsize=(12, 8))
    shap.summary_plot(
        shap_values, X_test,
        max_display=15,
        show=False
    )
    plt.title('SHAP — Dirección e impacto por feature', fontsize=13)
    plt.tight_layout()
    plt.show()

except ImportError:
    print('SHAP no disponible. Instalar con: pip install shap')


In [ ]:
# ============================================================
# 4.7 OPTIMIZACIÓN DE UMBRAL DE DECISIÓN
# ============================================================
# En problemas de churn, preferimos mayor recall (no perder churners)
# aunque baje la precisión. El umbral óptimo depende del costo de intervención.

y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.1, 0.9, 0.01)
recalls, precisions, f1s = [], [], []

for t in thresholds:
    y_pred_t = (y_proba_xgb >= t).astype(int)
    recalls.append(recall_score(y_test, y_pred_t, zero_division=0))
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))

optimal_t = thresholds[np.argmax(f1s)]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(thresholds, recalls,    color='#E8593C', lw=2, label='Recall')
ax.plot(thresholds, precisions, color='#7F77DD', lw=2, label='Precision')
ax.plot(thresholds, f1s,        color='#1D9E75', lw=2, label='F1-Score')
ax.axvline(optimal_t, color='gray', linestyle='--', alpha=0.8,
           label=f'Umbral óptimo F1: {optimal_t:.2f}')
ax.axvline(0.5, color='black', linestyle=':', alpha=0.4, label='Default (0.5)')
ax.set_xlabel('Umbral de decisión')
ax.set_ylabel('Métrica')
ax.set_title('Recall, Precision y F1 según umbral — XGBoost', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Métricas con umbral óptimo
y_pred_optimal = (y_proba_xgb >= optimal_t).astype(int)
print(f'=== Métricas con umbral = {optimal_t:.2f} ===')
print(f'Recall:    {recall_score(y_test, y_pred_optimal):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_optimal):.4f}')
print(f'F1-Score:  {f1_score(y_test, y_pred_optimal):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_proba_xgb):.4f}')


---
## Fase 5 — Evaluación desde la perspectiva de negocio


In [ ]:
# ============================================================
# 5.1 ANÁLISIS COSTO-BENEFICIO
# ============================================================
# Supuestos de negocio (ajustar según empresa real)
MONTHLY_REVENUE_PER_CUSTOMER = 65    # USD promedio
AVG_TENURE_MONTHS             = 24   # meses promedio de vida útil
CUSTOMER_LTV                  = MONTHLY_REVENUE_PER_CUSTOMER * AVG_TENURE_MONTHS
RETENTION_CAMPAIGN_COST       = 50   # USD por cliente contactado
RETENTION_SUCCESS_RATE        = 0.30  # 30% de clientes contactados retienen

cm = confusion_matrix(y_test, y_pred_optimal)
TN, FP, FN, TP = cm.ravel()

# Beneficio: clientes retenidos correctamente identificados
benefit = TP * RETENTION_SUCCESS_RATE * CUSTOMER_LTV

# Costo: campañas enviadas (TP + FP)
cost    = (TP + FP) * RETENTION_CAMPAIGN_COST

# Pérdida por no detectar churners reales (FN)
missed_revenue = FN * MONTHLY_REVENUE_PER_CUSTOMER * 6  # 6 meses de ingreso perdido

roi = (benefit - cost) / cost if cost > 0 else 0

print('=== ANÁLISIS COSTO-BENEFICIO (datos de test escalados a 12 meses) ===')
print(f'Clientes en riesgo detectados (TP):   {TP:,}')
print(f'Falsas alarmas (FP):                  {FP:,}')
print(f'Churners no detectados (FN):          {FN:,}')
print()
print(f'Beneficio estimado (retención):       ${benefit:,.0f}')
print(f'Costo de campañas:                    ${cost:,.0f}')
print(f'Ingreso perdido por FN:               ${missed_revenue:,.0f}')
print(f'Beneficio neto:                       ${benefit - cost:,.0f}')
print(f'ROI campañas:                         {roi:.1f}x')
print()
print(f'Meta ROI ≥ 3:1 → {"✅ ALCANZADA" if roi >= 3 else "❌ NO alcanzada"}')


In [ ]:
# ============================================================
# 5.2 CURVA DE GANANCIA (LIFT CURVE)
# ============================================================
# ¿Cuánto mejor que el azar es nuestro modelo?

# Ordenar por probabilidad descendente
gain_df = pd.DataFrame({
    'y_real':  y_test.values,
    'y_proba': y_proba_xgb
}).sort_values('y_proba', ascending=False).reset_index(drop=True)

gain_df['cumulative_churn']  = gain_df['y_real'].cumsum()
gain_df['pct_population']    = (gain_df.index + 1) / len(gain_df) * 100
gain_df['pct_churn_captured'] = gain_df['cumulative_churn'] / gain_df['y_real'].sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Curva de Ganancia
axes[0].plot(gain_df['pct_population'], gain_df['pct_churn_captured'],
             color='#7F77DD', lw=2.5, label='XGBoost')
axes[0].plot([0, 100], [0, 100], 'k--', alpha=0.4, label='Modelo aleatorio')
axes[0].fill_between(gain_df['pct_population'],
                     gain_df['pct_churn_captured'],
                     gain_df['pct_population'],
                     alpha=0.15, color='#7F77DD')

# Línea de referencia: con 30% de clientes contactados capturamos X% de churners
idx_30 = gain_df[gain_df['pct_population'] <= 30].index[-1]
captured_30 = gain_df.loc[idx_30, 'pct_churn_captured']
axes[0].annotate(f'Top 30% contactados\ncaptura {captured_30:.0f}% de churners',
                 xy=(30, captured_30), xytext=(45, captured_30 - 10),
                 fontsize=9, arrowprops=dict(arrowstyle='->', color='gray'))

axes[0].set_xlabel('% de clientes contactados')
axes[0].set_ylabel('% de churners capturados')
axes[0].set_title('Curva de Ganancia — XGBoost', fontsize=12)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Lift Chart
gain_df['lift'] = gain_df['pct_churn_captured'] / gain_df['pct_population']
axes[1].plot(gain_df['pct_population'], gain_df['lift'],
             color='#E8593C', lw=2.5)
axes[1].axhline(1, color='gray', linestyle='--', alpha=0.6, label='Lift = 1 (aleatorio)')
axes[1].set_xlabel('% de clientes contactados')
axes[1].set_ylabel('Lift')
axes[1].set_title('Curva de Lift — XGBoost', fontsize=12)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Análisis de Ganancia y Lift', fontsize=13)
plt.tight_layout()
plt.show()

lift_30 = gain_df.loc[idx_30, 'lift']
print(f'\nContactando al 30% de mayor riesgo:')
print(f'  → Se captura el {captured_30:.0f}% de todos los churners')
print(f'  → Lift = {lift_30:.2f}x mejor que selección aleatoria')


In [ ]:
# ============================================================
# 5.3 VALIDACIÓN DE KPIs DE NEGOCIO
# ============================================================
kpis = {
    'ROC-AUC ≥ 0.85':    roc_auc_score(y_test, y_proba_xgb) >= 0.85,
    'Recall ≥ 0.75':     recall_score(y_test, y_pred_optimal) >= 0.75,
    'ROI campañas ≥ 3x': roi >= 3,
    'Lift top 30% ≥ 2x': lift_30 >= 2
}

print('=== VALIDACIÓN DE KPIs DE NEGOCIO ===')
for kpi, met in kpis.items():
    status = '✅ CUMPLIDO' if met else '❌ NO cumplido'
    print(f'  {kpi}: {status}')

print(f'\nKPIs alcanzados: {sum(kpis.values())}/{len(kpis)}')


---
## Fase 6 — Conclusiones y Despliegue


In [ ]:
# ============================================================
# 6.1 PREDICCIÓN SOBRE NUEVOS CLIENTES (simulación de despliegue)
# ============================================================
import warnings
warnings.filterwarnings('ignore')

def predict_churn_risk(new_customers_df):
    """
    Función de inferencia lista para integrar en una API.
    Recibe un DataFrame con las mismas columnas que X_train.
    Retorna probabilidad de churn y segmento de riesgo.
    """
    proba = xgb.predict_proba(new_customers_df)[:, 1]
    risk  = pd.cut(proba,
                   bins=[0, 0.33, 0.66, 1.0],
                   labels=['Bajo', 'Medio', 'Alto'],
                   include_lowest=True)
    return pd.DataFrame({
        'churn_probability': proba.round(4),
        'risk_segment':      risk,
        'recommend_action':  pd.cut(proba,
                                    bins=[0, 0.33, 0.66, 1.0],
                                    labels=['Monitoreo', 'Oferta proactiva', 'Intervención urgente'],
                                    include_lowest=True)
    })

# Predicción sobre 10 clientes del set de prueba
sample = X_test.head(10)
predictions = predict_churn_risk(sample)
predictions['churn_real'] = y_test.head(10).values

print('=== PREDICCIONES SOBRE MUESTRA DE CLIENTES ===')
print(predictions.to_string(index=True))


In [ ]:
# ============================================================
# 6.2 GUARDAR MODELO FINAL
# ============================================================
import pickle

# Guardar modelo XGBoost y scaler
with open('xgb_churn_model.pkl', 'wb') as f:
    pickle.dump(xgb, f)

with open('scaler_churn.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Guardar lista de features para validación en producción
feature_names = X_train_res.columns.tolist()
with open('feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

print('✅ Modelo guardado: xgb_churn_model.pkl')
print('✅ Scaler guardado: scaler_churn.pkl')
print(f'✅ Features: {len(feature_names)} variables')


In [ ]:
# ============================================================
# 6.3 RESUMEN EJECUTIVO FINAL
# ============================================================
print('=' * 65)
print('          RESUMEN EJECUTIVO — PROYECTO CHURN CRISP-DM')
print('=' * 65)

print()
print('📊 DATOS')
print(f'   Dataset:   IBM Telco Customer Churn Extended')
print(f'   Registros: {len(df_model):,} clientes')
print(f'   Features originales: 21 → Features finales: {len(feature_names)}')
print(f'   Tasa de churn base: ~26%')

print()
print('🔧 FEATURE ENGINEERING')
print('   18 features nuevas creadas en 4 categorías:')
print('   • 5 temporales (tenure_cohort, charge_drift, ...)')
print('   • 4 de servicios (num_additional_services, ...)')
print('   • 1 score compuesto (churn_risk_score)')
print('   • 8 interacciones y transformaciones')

print()
print('🤖 MEJOR MODELO: XGBoost')
print(f'   ROC-AUC:   {roc_auc_score(y_test, y_proba_xgb):.4f}')
print(f'   Recall:    {recall_score(y_test, y_pred_optimal):.4f}')
print(f'   Precision: {precision_score(y_test, y_pred_optimal):.4f}')
print(f'   F1-Score:  {f1_score(y_test, y_pred_optimal):.4f}')
print(f'   Umbral óptimo: {optimal_t:.2f}')

print()
print('💼 IMPACTO DE NEGOCIO')
print(f'   ROI campañas: {roi:.1f}x')
print(f'   Beneficio neto estimado: ${benefit - cost:,.0f}')
print(f'   Contactando top 30% se captura {captured_30:.0f}% de churners')
print(f'   Lift: {lift_30:.2f}x mejor que campaña aleatoria')

print()
print('✅ KPIs alcanzados:', sum(kpis.values()), '/', len(kpis))
print('=' * 65)


---
## Referencias

- IBM Sample Data Sets: [IBM Telco Customer Churn](https://www.ibm.com/docs/en/cognos-analytics/12.0.x?topic=samples-telco-customer-churn)
- Kaggle Extended: [ylchang/telco-customer-churn-1113](https://www.kaggle.com/datasets/ylchang/telco-customer-churn-1113)
- CRISP-DM Reference Guide: [crisp-dm.eu](https://www.crisp-dm.eu)
- SMOTE: Chawla et al. (2002). *SMOTE: Synthetic Minority Over-sampling Technique*
- XGBoost: Chen & Guestrin (2016). *XGBoost: A Scalable Tree Boosting System*
- SHAP: Lundberg & Lee (2017). *A Unified Approach to Interpreting Model Predictions*
